# hold out method
<img src="../../images/st (1).png">
<img src="../../images/st (2).png">

# k fold method 
<img src="../../images/st (3).png">
<img src="../../images/st (4).png">

In [1]:
# hold out method 
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

class holdOut:
    def __init__(self):
        self.base_models=None
        self.meta_model=None
        
    def train(self,X,y):
        hold_X,base_X,hold_y,base_y=train_test_split(X,y,test_size=0.5)
        self.base_models=[LinearRegression(),DecisionTreeRegressor(max_depth=3)]
        for model in self.base_models:
            model.fit(base_X,base_y)
        
        hold_set=np.column_stack([model.predict(hold_X) for model in self.base_models])
        
        
        self.meta_model=RandomForestRegressor()
        self.meta_model.fit(hold_set,hold_y)

    def predict(self,X):
        hold_set=np.column_stack([model.predict(X) for model in self.base_models])
        return self.meta_model.predict(hold_set)
        

In [2]:
from sklearn.model_selection import KFold

class k_fold:
    def __init__(self,k):
        self.k=k
        self.base_models=None
        self.meta_model=None
        
    def train(self,X,y):
        kf = KFold(n_splits=self.k, shuffle=True, random_state=42)

        meta_X=[]
        meta_y=[]
        for train_index, val_index in kf.split(X):
            X_train,X_test,y_train,y_test=X[train_index],X[val_index],y[train_index],y[val_index]

            base_models=[LinearRegression(),DecisionTreeRegressor(max_depth=3)]
            for model in base_models:
                model.fit(X_train,y_train)
                
            meta_X.append(np.column_stack([model.predict(X_test) for model in base_models]))
            meta_y.append(y_test)
            
        self.meta_model=RandomForestRegressor()
        meta_X=np.vstack(meta_X)
        meta_y=np.concatenate(meta_y)
        self.meta_model.fit(meta_X,meta_y)

        self.base_models=[LinearRegression(),DecisionTreeRegressor(max_depth=3)]
        for model in self.base_models:
            model.fit(X,y)

    def predict(self,X):
        set_X=np.column_stack([model.predict(X) for model in self.base_models])
        return self.meta_model.predict(set_X)

# multilayer stacking

in this more than one layer are formed 

eg

Layer 0 (base models):  
  - LinearRegression
  - DecisionTreeRegressor

Layer 1 (first meta-model):
  - RandomForestRegressor
  - RandomForestRegressor

Layer 2 (final meta-model):
  - GradientBoostingRegressor

data flow  

Original features → Layer 0 (base models) → Layer 1 meta features  
Layer 1 meta features → Layer 1 model's → Layer 2 meta features  
Layer 2 meta features → Layer 2 model → final prediction


In [ ]:
# for classification it is similar

In [ ]:
# it wont work due to import error

from sklearn.ensemble import StackingRegressor


bg=BaggingRegressor(estimator=LinearRegression(),
bootstrap=False, bootstrap_features= True, max_features= 0.7, 
 max_samples= 1.0, n_estimators= 50)

clf=DecisionTreeRegressor(max_depth=11,criterion="friedman_mse")
gd=GradientBoostingRegressor(n_estimators=100,learning_rate=0.1,max_depth=4)
ad=AdaBoostRegressor(n_estimators=500,learning_rate=0.01)

rf=RandomForestRegressor(n_estimators=50,n_jobs=-1,max_depth=11,criterion="friedman_mse")

sr=StackingRegressor(estimators=[('ad',ad),('rf',rf),('bg',bg),('clf',clf)],final_estimator=gd,n_jobs=-1,cv=5,verbose=1)

scores = cross_val_score(sr, X=train[:,:-1],y=train[:,-1], cv=10)

print("Mean Cross-validated accuracy:", scores.mean())
print("Best Cross-validated accuracy:", scores.max())

In [ ]:
# 2 layer

from sklearn.model_selection import KFold

class k_fold_2_layer:
    def __init__(self,k):
        self.k=k
        self.base_models=None
        self.meta_model=None
        self.base_models2=None
        
    def train(self,X,y):
        kf = KFold(n_splits=self.k, shuffle=True, random_state=42)

        meta_X=[]
        meta_y=[]

        # layer 1
        for train_index, val_index in kf.split(X):
            X_train,X_test,y_train,y_test=X[train_index],X[val_index],y[train_index],y[val_index]

            base_models=[LinearRegression(),DecisionTreeRegressor(max_depth=3)]
            for model in base_models:
                model.fit(X_train,y_train)
                
            meta_X.append(np.column_stack([model.predict(X_test) for model in base_models]))
            meta_y.append(y_test)
        meta_X=np.vstack(meta_X)
        meta_y=np.concatenate(meta_y)
        

        # layer 2
        meta_X_=[]
        meta_y_=[]
        for train_index, val_index in kf.split(meta_X):
            X_train,X_test,y_train,y_test=X[train_index],X[val_index],meta_y[train_index],meta_y[val_index]

            base_models=[LinearRegression(),DecisionTreeRegressor(max_depth=3)]
            for model in base_models:
                model.fit(X_train,y_train)
                
            meta_X_.append(np.column_stack([model.predict(X_test) for model in base_models]))
            meta_y_.append(y_test)
            
        meta_X_=np.vstack(meta_X_)
        meta_y_=np.concatenate(meta_y_)
        self.meta_model=RandomForestRegressor()
        self.meta_model.fit(meta_X_,meta_y_)

        self.base_models=[LinearRegression(),DecisionTreeRegressor(max_depth=3)]
        self.base_models2=[LinearRegression(),DecisionTreeRegressor(max_depth=3)]
        for model in self.base_models:
            model.fit(X,y)

        layer1_out = np.column_stack([model.predict(X) for model in self.base_models])
        for model in self.base_models2:
            model.fit(layer1_out,y)

    def predict(self,X):
        layer1_out = np.column_stack([model.predict(X) for model in self.base_models])
        layer2_out = np.column_stack([model.predict(layer1_out) for model in self.base_models2])
        return self.meta_model.predict(layer2_out)
